### FastAPI

#### First /chat Endpoint

- Key Points:

    - Defines a route that handles chat requests
    
    - Uses HTTP methods (GET, POST, etc.)
    
    - Returns JSON responses by default

In [ ]:
from fastapi import FastAPI

app = FastAPI()

@app.get("/home")
def home_page():

    return {"message": "Welcome to the Main page"}

#### Pydantic Request/Response Models

- Key Points:

    - Data validation using Python type hints
    
    - Automatic parsing of JSON requests

    - Response modeling for consistent output

    - Validation errors are automatically handled

In [2]:
# without pydantic

In [ ]:
from fastapi import FastAPI

app = FastAPI()

@app.post("/chat")
def chat(text: str) -> str:

    return {"message": f"{text}"}

In [ ]:
# with pydantic

In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI()

# --------------------------------------------------------
class ChatRequest(BaseModel):
    message: str


class ChatResponse(BaseModel):
    reply: str
    token_used: int

# --------------------------------------------------------
@app.post("/chat", response_model = ChatResponse)
def chat(request: ChatRequest):

    response = f"you said : {request.message}"

    return ChatResponse(
        reply = response,
        token_used = len(request.message.split())
    )

#### Dependency Injection

- Key Points:

    - Reusable components injected into endpoints

    - Automatic resolution of dependencies

    - Great for database connections, auth, caching

In [ ]:
from fastapi import FastAPI, Depends
from pydantic import BaseModel

app = FastAPI()

# ----------------------- Validation ------------------------
class ChatRequest(BaseModel):
    message: str


class ChatResponse(BaseModel):
    reply: str
    token_used: int

# ---------------------- Dependecy  -----------------------
def validate_msg(request: ChatRequest):

    if len(request.message) == 0:
        raise Exception("Message shouldn't be empty")

    if len(request.message) > 100:
        raise Exception("Message is too long")

    return {"message": request.message, "length": len(request.message)}


# --------------------------------------------------------
# FastAPI executes validate_msg() first.
# If validation succeeds, chat() is executed.


@app.post("/chat", response_model = ChatResponse)
def chat(
    request: ChatRequest,
    msg: dict = Depends(validate_msg)               # dependency
):

    print(msg["length"])   # استفاده از خروجی Dependency

    response = f"you said : {msg['message']}"

    return ChatResponse(
        reply=response,
        token_used=len(msg["message"].split())
    )